In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
conn = sqlite3.connect("superstore.db")

In [4]:
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM superstore_raw LIMIT 5", conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
query = """
CREATE TABLE customers AS
SELECT DISTINCT
    "Customer ID" AS Customer_ID,
    "Customer Name" AS Customer_Name,
    Segment
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

In [6]:
query = """
CREATE TABLE orders AS
SELECT DISTINCT
    "Order ID" AS Order_ID,
    "Order Date" AS Order_Date,
    "Ship Date" AS Ship_Date,
    "Ship Mode" AS Ship_Mode,
    "Customer ID" AS Customer_ID,
    Sales,
    Quantity,
    Profit
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

In [7]:
query = """
CREATE TABLE products AS
SELECT DISTINCT
    "Product ID" AS Product_ID,
    "Product Name" AS Product_Name,
    Category,
    "Sub-Category" AS Sub_Category
FROM superstore_raw;
"""

conn.execute(query)
conn.commit()

In [8]:
query = """
SELECT *
FROM orders
WHERE Sales >
(
    SELECT AVG(Sales)
    FROM orders
);
"""

pd.read_sql(query, conn)

,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Sales,Quantity,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,261.9600,2,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.9400,3,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.5775,5,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,907.1520,6,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,1706.1840,9,85.3092
...,...,...,...,...,...,...,...,...
2354,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,271.9600,5,27.1960
2355,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,249.5840,2,31.1980
2356,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,437.4720,14,153.1152
2357,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,258.5760,2,19.3932


In [9]:
query = """
SELECT *
FROM orders o
WHERE Sales =
(
    SELECT MAX(Sales)
    FROM orders
    WHERE Customer_ID = o.Customer_ID
);
"""

pd.read_sql(query, conn)

,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Sales,Quantity,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.9400,3,219.5820
1,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.5775,5,-383.0310
2,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,1706.1840,9,85.3092
3,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,1044.6300,3,240.2649
4,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,3083.4300,7,-1665.0522
...,...,...,...,...,...,...,...,...
790,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,1087.9360,8,353.5792
791,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,2799.9600,5,944.9865
792,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,2405.2000,8,793.7160
793,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,735.9800,2,331.1910


In [10]:
query = """
WITH customer_sales AS
(
    SELECT Customer_ID,
           SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT *
FROM customer_sales;
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,AA-10315,5563.560
1,AA-10375,1056.390
2,AA-10480,1790.512
3,AA-10645,5086.935
4,AB-10015,886.156
...,...,...
788,XP-21865,2374.658
789,YC-21895,5454.350
790,YS-21880,6720.444
791,ZC-21910,8025.707


In [11]:
query = """
WITH customer_sales AS
(
    SELECT Customer_ID,
           SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT *
FROM customer_sales
WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
);
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


In [12]:
query = """
SELECT
    Customer_ID,
    SUM(Sales) AS Total_Sales,
    RANK() OVER(ORDER BY SUM(Sales) DESC) AS Sales_Rank
FROM orders
GROUP BY Customer_ID;
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales,Sales_Rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


In [13]:
query = """
SELECT
    Customer_ID,
    Order_ID,
    Sales,
    ROW_NUMBER() OVER
    (
        PARTITION BY Customer_ID
        ORDER BY Sales DESC
    ) AS Row_Num
FROM orders;
"""

pd.read_sql(query, conn)

,Customer_ID,Order_ID,Sales,Row_Num
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2014-128055,673.568,2
2,AA-10315,CA-2016-103982,431.976,3
3,AA-10315,CA-2017-147039,362.940,4
4,AA-10315,CA-2014-128055,52.980,5
...,...,...,...,...
9988,ZD-21925,CA-2017-141481,61.440,5
9989,ZD-21925,CA-2014-143336,22.720,6
9990,ZD-21925,US-2016-147991,16.720,7
9991,ZD-21925,CA-2016-152471,15.984,8


In [14]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales,
        RANK() OVER(ORDER BY SUM(Sales) DESC) AS Sales_Rank
    FROM orders
    GROUP BY Customer_ID
)
SELECT *
FROM customer_sales
WHERE Sales_Rank <= 3;
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales,Sales_Rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


In [15]:
query = """
WITH customer_sales AS
(
    SELECT
        Customer_ID,
        SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT
    c.Customer_Name,
    cs.Total_Sales,
    RANK() OVER(ORDER BY cs.Total_Sales DESC) AS Sales_Rank
FROM customer_sales cs
JOIN customers c
ON cs.Customer_ID = c.Customer_ID;
"""

pd.read_sql(query, conn)

,Customer_Name,Total_Sales,Sales_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [16]:
query = """
SELECT
    c.Customer_Name,
    SUM(o.Sales) AS Total_Sales
FROM customers c
JOIN orders o
ON c.Customer_ID = o.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Total_Sales DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer_Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571


In [17]:
query = """
SELECT
    c.Customer_Name,
    SUM(o.Sales) AS Total_Sales
FROM customers c
JOIN orders o
ON c.Customer_ID = o.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Total_Sales ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer_Name,Total_Sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328


In [21]:
query = """
SELECT
    c.Customer_Name,
    SUM(o.Sales) AS Total_Sales
FROM customers c
JOIN orders o
ON c.Customer_ID = o.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Total_Sales ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer_Name,Total_Sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328


In [18]:
query = """
SELECT
    c.Customer_Name
FROM customers c
JOIN orders o
ON c.Customer_ID = o.Customer_ID
GROUP BY c.Customer_Name
HAVING COUNT(DISTINCT o.Order_ID) = 1;
"""

pd.read_sql(query, conn)

,Customer_Name
0,Anemone Ratner
1,Anthony O'Donnell
2,Carl Jackson
3,Jenna Caffey
4,Jocasta Rupert
5,Lela Donovan
6,Mitch Gastineau
7,Patricia Hirasaki
8,Ricardo Emerson
9,Roland Murray


In [19]:
query = """
WITH customer_sales AS
(
    SELECT Customer_ID,
           SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)
SELECT *
FROM customer_sales
WHERE Total_Sales >
(
    SELECT AVG(Total_Sales)
    FROM customer_sales
);
"""

pd.read_sql(query, conn)

,Customer_ID,Total_Sales
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


In [20]:
query = """
SELECT
    Customer_ID,
    MAX(Sales) AS Highest_Order_Value
FROM orders
GROUP BY Customer_ID;
"""

pd.read_sql(query, conn)

,Customer_ID,Highest_Order_Value
0,AA-10315,3930.072
1,AA-10375,499.980
2,AA-10480,479.970
3,AA-10645,1323.900
4,AB-10015,341.960
...,...,...
788,XP-21865,337.088
789,YC-21895,2934.330
790,YS-21880,2793.528
791,ZC-21910,1516.200
